In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/Nikhil3654/generalizable-llm-planning.git"
BRANCH = "feature/benchmark-index"

WORKING_DIR = Path("/kaggle/working")
REPO_DIR = WORKING_DIR / "generalizable-llm-planning"
KAGGLE_INPUT = Path("/kaggle/input")


In [ ]:
if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "origin", BRANCH], check=True)
else:
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO_DIR)],
        check=True,
    )

repo_path = str(REPO_DIR)
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)

print("Repository ready:", REPO_DIR)


In [ ]:
matches = list(
    KAGGLE_INPUT.rglob("ipc-2000/domains/blocks-strips-untyped/domain.pddl")
)

if not matches:
    raise FileNotFoundError(
        "Could not locate the IPC dataset under /kaggle/input. "
        "Attach the IPC PDDL dataset using Add Input."
    )

if len(matches) > 1:
    print("Multiple IPC dataset matches found:")
    for match in matches:
        print(" -", match)
    print("\nUsing the first match.")

reference_domain = matches[0]

# Expected structure:
# DATASET_ROOT/ipc-2000/domains/blocks-strips-untyped/domain.pddl
DATASET_ROOT = reference_domain.parents[3]

print("Dataset root:", DATASET_ROOT)
print("Reference domain:", reference_domain)


In [ ]:
from src.benchmark import discover_domains

domains = discover_domains(DATASET_ROOT)

print("Discovered domain variants:", len(domains))
print()

for domain in domains[:20]:
    print(
        domain["name"],
        "| instances:",
        domain["num_instances"],
        "|",
        domain["domain_file"],
    )


In [ ]:
import pandas as pd

domain_df = pd.DataFrame([
    {
        "domain_variant": item["name"],
        "num_instances": item["num_instances"],
        "domain_file": str(item["domain_file"].relative_to(DATASET_ROOT)),
    }
    for item in domains
])

print("Rows:", len(domain_df))
display(domain_df.head(20))


In [ ]:
def competition_from_path(relative_path):
    for part in Path(relative_path).parts:
        if part.startswith("ipc-"):
            return part
    return "unknown"

domain_df["competition"] = domain_df["domain_file"].apply(
    competition_from_path
)

competition_summary = (
    domain_df
    .groupby("competition", as_index=False)
    .agg(
        domain_variants=("domain_variant", "count"),
        total_instances=("num_instances", "sum"),
    )
    .sort_values("competition")
)

display(competition_summary)


In [ ]:
largest_domains = (
    domain_df
    .sort_values("num_instances", ascending=False)
    .head(20)
)

display(largest_domains)


In [ ]:
from src.benchmark import build_benchmark_index

benchmark_df = build_benchmark_index(DATASET_ROOT)

print("Total benchmark problems:", len(benchmark_df))
num_domain_variants = (
    benchmark_df[
        ["competition", "domain_variant"]
    ]
    .drop_duplicates()
    .shape[0]
)

print(
    "Competition/domain variants:",
    num_domain_variants
)
print("Competitions:", benchmark_df["competition"].nunique())

display(benchmark_df.head(20))


In [ ]:
OUTPUT_PATH = WORKING_DIR / "benchmark_index.csv"

benchmark_df.to_csv(
    OUTPUT_PATH,
    index=False,
)

print("Saved:", OUTPUT_PATH)
print("Rows:", len(benchmark_df))


In [ ]:
assert len(benchmark_df) > 0
assert benchmark_df["problem"].notna().all()
assert benchmark_df["domain_file"].notna().all()
assert benchmark_df["problem_file"].notna().all()

duplicates = benchmark_df.duplicated(
    subset=["competition", "domain_variant", "problem"]
).sum()

print("Duplicate benchmark rows:", duplicates)

display(
    benchmark_df.groupby(
        ["competition", "domain_variant"],
        as_index=False
    ).size().head(30)
)
